Same import from Gutenburg

In [ ]:
import torch
import os

destination_path = os.path.join(os.getcwd(), "dracula.txt")
torch.hub.download_url_to_file('https://www.gutenberg.org/cache/epub/345/pg345.txt', destination_path)
with open(destination_path, 'r') as file:
    corpus = file.read()

100%|██████████| 870k/870k [00:00<00:00, 2.72MB/s]


nltk tokenizer and stopwords to clean corpus before byte pair encoding

In [ ]:
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

#regex (⓿_⓿)
tokenizer = RegexpTokenizer(r'\w+')
splitCorpus = [w for w in tokenizer.tokenize(corpus) if w.lower() not in stop_words]

print(splitCorpus)

['Project', 'Gutenberg', 'eBook', 'Dracula', 'ebook', 'use', 'anyone', 'anywhere', 'United', 'States', 'parts', 'world', 'cost', 'almost', 'restrictions', 'whatsoever', 'may', 'copy', 'give', 'away', 'use', 'terms', 'Project', 'Gutenberg', 'License', 'included', 'ebook', 'online', 'www', 'gutenberg', 'org', 'located', 'United', 'States', 'check', 'laws', 'country', 'located', 'using', 'eBook', 'Title', 'Dracula', 'Author', 'Bram', 'Stoker', 'Release', 'date', 'October', '1', '1995', 'eBook', '345', 'recently', 'updated', 'November', '12', '2023', 'Language', 'English', 'Credits', 'Chuck', 'Greif', 'Online', 'Distributed', 'Proofreading', 'Team', 'START', 'PROJECT', 'GUTENBERG', 'EBOOK', 'DRACULA', 'DRACULA', '_by_', 'Bram', 'Stoker', 'Illustration', 'colophon', 'NEW', 'YORK', 'GROSSET', 'DUNLAP', '_Publishers_', 'Copyright', '1897', 'United', 'States', 'America', 'according', 'Act', 'Congress', 'Bram', 'Stoker', '_All', 'rights', 'reserved', '_', 'PRINTED', 'UNITED', 'STATES', 'COUNTRY

Byte pair encoding functions

In [ ]:
from collections import Counter, defaultdict


In [ ]:
def PairFreq(vocab):
  #count frequency of pairs
  pairs = Counter()
  for word, freq in vocab.items():
    symbols = word.split() # cut up "w o r d" or "wo r d"
    for i in range(len(symbols) - 1):
      pairs[(symbols[i], symbols[i+1])] += freq
  return pairs #('a', 'b'): 100, ('b', 'c'): 50, .....

In [ ]:
def Merge(pair, vocab):
  #merge all instances of specific pair
  bigram = " ".join(pair) # w o
  replacement = "".join(pair) # wo
  new_vocab = {}
  for word in vocab: # w o r d
    new_word = word.replace(bigram, replacement) # wo r d
    new_vocab[new_word] = vocab[word]
  return new_vocab

In [ ]:
def PrePear(corpus, numMerges):
  #learn merges
  vocab = Counter([" ".join(word) for word in corpus])#split each word like t h i s and counter it

  merges = []
  for _ in range(numMerges):
    pairs = PairFreq(vocab)
    if not pairs: # ╯︿╰
      break
    best = max(pairs, key=pairs.get)
    vocab = Merge(best, vocab)
    merges.append(best)
  return merges



In [92]:
def BuildVocab(merges):
  #get valid token list from merge list
  vocab = set()

  # add all base symbols
  for a, b in merges:
    vocab.add(a)
    vocab.add(b)

  # add all merged tokens
  for a, b in merges:
    vocab.add(a + b)

  return vocab

In [94]:
def BitePear(word, vocab):
  #byte pair single word
  tokens = []
  i = 0
  while i < len(word):
    #find the longest token that matches starting at position i
    match = None
    for j in range(len(word), i, -1):  #try longer substrings first
      candidate = word[i:j]
      if candidate in vocab:
        match = candidate
        tokens.append(match)
        i = j
        break
    if match is None:  #no valid token, fall back to single char
      tokens.append(word[i])
      i += 1
  return tokens

In [102]:
def BitePears(corpus, vocab):
  #byte pair entire text
  tokenized = []
  for word in corpus:
    tokenized.extend(BitePear(word, vocab))
  return tokenized

Learn 1,000 pairs from Dracula

In [58]:
merges = PrePear(splitCorpus, 1000) #learn tokens from our prepped corpus up to 1,000 merges
print(merges)


[('i', 'n'), ('e', 'r'), ('e', 'd'), ('e', 'n'), ('s', 't'), ('in', 'g'), ('e', 'a'), ('o', 'u'), ('a', 'n'), ('o', 'n'), ('t', 'h'), ('o', 'r'), ('e', 's'), ('a', 'r'), ('l', 'l'), ('s', 'e'), ('l', 'e'), ('a', 't'), ('r', 'e'), ('a', 'y'), ('r', 'i'), ('o', 'm'), ('g', 'h'), ('l', 'd'), ('o', 'w'), ('l', 'i'), ('o', 'o'), ('s', 'h'), ('t', 'i'), ('l', 'y'), ('c', 'e'), ('c', 'h'), ('a', 'l'), ('gh', 't'), ('i', 'd'), ('k', 'e'), ('en', 't'), ('s', 'a'), ('m', 'e'), ('e', 'v'), ('e', 't'), ('r', 'o'), ('i', 't'), ('l', 'a'), ('r', 'a'), ('ou', 'ld'), ('v', 'e'), ('u', 'n'), ('on', 'e'), ('u', 'r'), ('c', 'k'), ('se', 'e'), ('e', 'l'), ('u', 's'), ('k', 'n'), ('m', 'a'), ('s', 'i'), ('c', 'a'), ('e', 'p'), ('l', 'o'), ('c', 't'), ('d', 'i'), ('t', 'er'), ('f', 'i'), ('o', 'p'), ('a', 's'), ('g', 'o'), ('w', 'h'), ('i', 'ght'), ('sa', 'id'), ('u', 't'), ('an', 'd'), ('s', 'p'), ('q', 'u'), ('w', 'ay'), ('w', 'or'), ('kn', 'ow'), ('th', 'ing'), ('ar', 'd'), ('b', 'le'), ('t', 'a'), ('in'

Build vocab of valid tokens from merge list

In [93]:
tokens = BuildVocab(merges)
print(tokens)

{'oundation', 'behind', 'pretty', 'moving', 'hyp', 'awk', 'ther', 'experi', 'please', 'llow', 'kn', 'necessary', 'ild', 'r', 'pe', 'knows', 'cap', 'straight', 'lea', 'JO', 'words', 'cur', 'follow', 'help', 'Dr', 'sa', 'pla', 'tom', 'three', 'upset', 'af', 'also', 'fixed', 'sunset', 'fier', 'led', 'HARK', 'hor', 'ri', 'took', 'iss', 'found', 'tomb', 'youn', 'saying', 'suc', 'chy', 'fted', 'account', 'advan', 'In', 'bea', 'cept', 'attendant', 'trouble', 'ton', 'driver', 'making', 'per', 'int', 'away', 'strength', 'sorrow', 'gh', 'rush', 'pause', 'om', 'op', 'z', 'bit', 'mit', 'lit', 'su', 'rib', 'answ', 'love', 'Oh', 'sear', 'Slo', 'ti', 'Diary', 'attendan', 'said', 'ho', 'early', 'lifted', 'don', 'thing', 'flowers', 'where', 'lled', 'fort', 'kni', 'ctober', 'laid', 'ise', 'T', 'ross', 'ordinary', 'Hel', 'ice', 'somehow', 'sing', 'begin', 'sou', 'passed', 'lun', 'Come', 'ter', 'un', 'recei', 'go', 'mid', 'Westen', 'September', 'speak', 'cer', 'together', 'ject', 'present', 'brave', 'Arth

Dracula get his own dedicated token, but not Transylvania.

In [95]:
BitePear("Dracula", tokens)

['Dracula']

In [98]:
BitePear("Transylvania", tokens)

['Tran', 'sy', 'l', 'van', 'ia']

Tokenize the corpus

In [103]:
tokenized = BitePears(splitCorpus, tokens)

Most frequent tokens.
Mostly word fragments of course, with some words like "though" and "must" popping up. "Van" and "Helsing" are the first tokens that pop up that have specific relevance to the novel. Oddly enough "Van" appears three more times than "Helsing", since they likey didn't have large automobiles back then, I'll assume they refer to the good professor informally a number of times.

In [104]:
Counter(tokenized).most_common()

[('s', 3640),
 ('n', 2751),
 ('d', 2301),
 ('t', 2118),
 ('ed', 1785),
 ('e', 1754),
 ('st', 1670),
 ('ing', 1669),
 ('y', 1647),
 ('de', 1410),
 ('o', 1068),
 ('r', 1052),
 ('ar', 993),
 ('lo', 984),
 ('ly', 977),
 ('ce', 969),
 ('ma', 923),
 ('l', 908),
 ('me', 893),
 ('er', 831),
 ('te', 798),
 ('re', 788),
 ('_', 769),
 ('ll', 751),
 ('le', 746),
 ('ne', 726),
 ('si', 725),
 ('g', 695),
 ('one', 681),
 ('go', 670),
 ('on', 660),
 ('k', 642),
 ('ti', 635),
 ('es', 634),
 ('us', 614),
 ('se', 613),
 ('al', 612),
 ('said', 568),
 ('la', 559),
 ('p', 558),
 ('man', 542),
 ('i', 533),
 ('at', 509),
 ('see', 503),
 ('w', 500),
 ('could', 490),
 ('fe', 482),
 ('en', 477),
 ('h', 476),
 ('wi', 470),
 ('time', 467),
 ('or', 464),
 ('c', 462),
 ('sh', 449),
 ('must', 448),
 ('ro', 439),
 ('in', 434),
 ('fa', 432),
 ('m', 430),
 ('ct', 427),
 ('would', 427),
 ('di', 427),
 ('shall', 427),
 ('ri', 425),
 ('an', 422),
 ('know', 416),
 ('though', 414),
 ('b', 410),
 ('po', 407),
 ('be', 406),
 (

Largest tokens.
1000 merges gets us a good number of words, even covering things like "Piccadilly" and "Godalming". Initial test values were 100,000 merges, which was now obviously overkill.

In [105]:
unique = set(tokenized)
sorted(unique, key=len, reverse=True)

['opportunity',
 'everything',
 'experience',
 'Foundation',
 'electronic',
 'difficulty',
 'determined',
 'Piccadilly',
 'churchyard',
 'necessary',
 'attendant',
 'September',
 'whispered',
 'satisfied',
 'moonlight',
 'certainly',
 'Godalming',
 'afternoon',
 'beautiful',
 'yesterday',
 'beginning',
 'knowledge',
 'evidently',
 'breakfast',
 'wonderful',
 'centuries',
 'continued',
 'Professor',
 'copyright',
 'questions',
 'seemingly',
 'conscious',
 'Gutenberg',
 'straight',
 'strength',
 'ordinary',
 'together',
 'possible',
 'Westenra',
 'finished',
 'opportun',
 'children',
 'cheerful',
 'suddenly',
 'remember',
 'surprise',
 'forehead',
 'dreadful',
 'actually',
 'crucifix',
 'anywhere',
 'expected',
 'Renfield',
 'terrible',
 'received',
 'telegram',
 'horrible',
 'anything',
 'Jonathan',
 'complete',
 'Bistritz',
 'account',
 'trouble',
 'flowers',
 'somehow',
 'present',
 'writing',
 'arrived',
 'nothing',
 'pointed',
 'instant',
 'Dracula',
 'strange',
 'running',
 'certai

Potential glitch tokens?
Token pairs that are a part of a larger pair, and never used by themselves...

In [111]:
unused = tokens - unique
print(unused)

{'Joh', 'oundation', 'enou', 'Westen', 'loor', 'Godal', 'yester', 'peop', 'proo', 'glas', 'cadilly', 'lear', 'Galat', 'cessary', 'gir', 'uty', 'surp', 'istrit', 'Lu', 'lowers', 'HAR', 'Jonath', 'centuri', 'electron', '_Let', 'cep', 'rong', 'Slo', 'Carfa', 'ispered', 'Arth', 'CHAP', 'Lon', 'Dracu', 'attendan', 'anx', 'swep', 'esten', 'ften', 'than', 'HARK', 'sud', 'Sep', 'Quin', 'Than', 'q', 'Bistrit', 'Pic', 'Hawk', 'nex', 'Jour', 'dinary', 'ctober', 'chy', 'eward', 'Var', 'mined', 'uten', 'umber', 'churchy', 'tendan', 'shud', 'youn', 'usu', 'fron', 'alat', 'Septe', 'riend', 'riting', 'issed', 'ilst', 'opport', 'cey', 'Jon', 'Guten'}
